# Práctica N.° 06 — Pilas, Colas y Deque
**Balanceo, Evaluación de Expresiones y BFS — Python**

Universidad Nacional del Altiplano · Escuela Profesional de Ingeniería de Sistemas
Curso: Algoritmos y Estructuras de Datos — SIS210 · Semana 6

> Las celdas deben ejecutarse **en orden**: la Actividad 6 reutiliza funciones de las Actividades 2 y 3.
> La Actividad 5 (C++) está en el archivo `Practica_S06_PilasColasDeque.cpp`.

## Actividad 1 — `PilaEnlazada` y `ColaEnlazada` desde cero
Objetivo: implementar Pila y Cola con el `Nodo` de la Práctica 5. Todas las operaciones son O(1).

In [1]:
# Reutilizamos Nodo de la Práctica 5
from dataclasses import dataclass
from typing import Optional, Any

@dataclass
class Nodo:
    dato: Any
    siguiente: 'Optional[Nodo]' = None

class PilaEnlazada:
    def __init__(self):
        self._tope: Optional[Nodo] = None
        self._n: int = 0

    def push(self, dato) -> None:
        # Crear Nodo, apuntar al tope actual, actualizar tope  -> O(1)
        self._tope = Nodo(dato, self._tope)
        self._n += 1

    def pop(self) -> Any:
        # Verificar vacía, guardar dato del tope, actualizar tope, retornar dato  -> O(1)
        if self.es_vacia(): raise IndexError('Pila vacía')
        dato = self._tope.dato
        self._tope = self._tope.siguiente
        self._n -= 1
        return dato

    def peek(self) -> Any:
        if self.es_vacia(): raise IndexError('Pila vacía')
        return self._tope.dato

    def es_vacia(self) -> bool: return self._n == 0
    def __len__(self) -> int:   return self._n

class ColaEnlazada:
    def __init__(self):
        self._cabeza: Optional[Nodo] = None
        self._cola: Optional[Nodo] = None  # puntero tail para O(1) al final
        self._n = 0

    def enqueue(self, dato) -> None:
        # Insertar al final usando el puntero _cola  -> O(1)
        nuevo = Nodo(dato)
        if self.es_vacia():
            self._cabeza = nuevo
        else:
            self._cola.siguiente = nuevo
        self._cola = nuevo
        self._n += 1

    def dequeue(self) -> Any:
        # Retirar de la cabeza  -> O(1)
        if self.es_vacia(): raise IndexError('Cola vacía')
        dato = self._cabeza.dato
        self._cabeza = self._cabeza.siguiente
        if self._cabeza is None:      # la cola quedó vacía: soltar también el tail
            self._cola = None
        self._n -= 1
        return dato

    def frente(self) -> Any:
        if self.es_vacia(): raise IndexError('Cola vacía')
        return self._cabeza.dato

    def es_vacia(self) -> bool: return self._n == 0
    def __len__(self) -> int:   return self._n

# Pruebas
p = PilaEnlazada()
for v in [10,20,30]: p.push(v)
assert p.peek() == 30
assert p.pop()  == 30
assert len(p)   == 2

c = ColaEnlazada()
for v in [10,20,30]: c.enqueue(v)
assert c.frente()  == 10
assert c.dequeue() == 10
assert len(c)      == 2

# Pruebas adicionales: orden LIFO/FIFO, vaciado completo y reutilización
assert [p.pop(), p.pop()] == [20, 10] and p.es_vacia()
assert [c.dequeue(), c.dequeue()] == [20, 30] and c.es_vacia()
c.enqueue(99); assert c.frente() == 99 and len(c) == 1   # el tail se reinició bien
assert c.dequeue() == 99 and c.es_vacia()
for op in (p.pop, p.peek, c.dequeue, c.frente):          # sobre estructuras vacías
    try: op(); raise AssertionError('debía lanzar IndexError')
    except IndexError: pass
print('Pila y Cola: OK ✓')

Pila y Cola: OK ✓


## Actividad 2 — Balanceo de paréntesis con traza
Objetivo: implementar el algoritmo de balanceo (O(n)) con traza completa del estado de la pila.

In [2]:
def esta_balanceado(s: str, verbose: bool = False) -> bool:
    """
    Verifica si la cadena s tiene paréntesis/corchetes/llaves correctamente anidados.
    Si verbose=True, imprime la traza de la pila en cada paso.
    """
    pila = []
    pares = {')': '(', ']': '[', '}': '{'}

    for i, c in enumerate(s):
        if c in '([{':
            pila.append(c)
            if verbose: print(f'  [{i}] push({c!r}) → pila={pila}')
        elif c in ')]}':
            if not pila or pila[-1] != pares[c]:
                if verbose: print(f'  [{i}] pop({c!r}) → MISMATCH o pila vacía')
                return False
            pila.pop()
            if verbose: print(f'  [{i}] pop() ← {c!r} → pila={pila}')

    resultado = len(pila) == 0
    if verbose: print(f'  Pila final: {pila} → {"VÁLIDO" if resultado else "INVÁLIDO"}')
    return resultado

# Casos de prueba
casos = [
    ('([{}])',    True),
    ('([)]',     False),
    ('{[()]}',   True),
    ('(((',       False),
    ('',          True),
]
for s, esperado in casos:
    resultado = esta_balanceado(s)
    estado = '✓' if resultado == esperado else '✗ ERROR'
    assert resultado == esperado, f'Falló el caso {s!r}'
    print(f'{s!r:15} → {resultado}  {estado}')

# Traza completa de '([{}])'
print('\nTraza para "([{}])":')
esta_balanceado('([{}])', verbose=True)

'([{}])'        → True  ✓
'([)]'          → False  ✓
'{[()]}'        → True  ✓
'((('           → False  ✓
''              → True  ✓

Traza para "([{}])":
  [0] push('(') → pila=['(']
  [1] push('[') → pila=['(', '[']
  [2] push('{') → pila=['(', '[', '{']
  [3] pop() ← '}' → pila=['(', '[']
  [4] pop() ← ']' → pila=['(']
  [5] pop() ← ')' → pila=[]
  Pila final: [] → VÁLIDO


## Actividad 3 — Evaluación postfix y conversión Shunting-Yard
Objetivo: evaluar expresiones en notación postfix (RPN) y convertir infix → postfix.

**Corrección principal:** en el diccionario `ops`, la clave de la suma estaba escrita como `','` en lugar de `'+'`.

In [3]:
def evaluar_postfix(expr: str) -> float:
    """Evalúa expresión en notación postfix (RPN). Tokens separados por espacio."""
    pila = []
    ops = {'+': lambda a,b: a+b, '-': lambda a,b: a-b,      # CORREGIDO: antes era ','
           '*': lambda a,b: a*b, '/': lambda a,b: a/b}
    for token in expr.split():
        if token in ops:
            if len(pila) < 2:
                raise ValueError(f'Expresión inválida: faltan operandos para {token!r}')
            b, a = pila.pop(), pila.pop()  # b es el segundo operando
            pila.append(ops[token](a, b))
        else:
            pila.append(float(token))
    if len(pila) != 1:
        raise ValueError(f'Expresión inválida: pila tiene {len(pila)} elementos al final')
    return pila[0]

def infix_a_postfix(expr: str) -> str:
    """Algoritmo Shunting-Yard: convierte infix con paréntesis a postfix."""
    precedencia = {'+':1, '-':1, '*':2, '/':2}
    salida, pila = [], []
    for token in expr.split():
        if token.lstrip('-').replace('.','').isdigit():  # número
            salida.append(token)
        elif token in precedencia:  # operador
            while (pila and pila[-1] in precedencia and
                   precedencia[pila[-1]] >= precedencia[token]):
                salida.append(pila.pop())
            pila.append(token)
        elif token == '(': pila.append(token)
        elif token == ')':
            while pila and pila[-1] != '(': salida.append(pila.pop())
            if not pila:
                raise ValueError("Paréntesis no balanceados: sobra ')'")
            pila.pop()  # descartar '('
        else:
            raise ValueError(f'Token no reconocido: {token!r} (separe los tokens con espacios)')
    while pila:
        op = pila.pop()
        if op == '(':
            raise ValueError("Paréntesis no balanceados: sobra '('")
        salida.append(op)
    return ' '.join(salida)

# Verificación
expresiones = [
    ('3 4 + 5 *',   35.0),   # postfix directo: (3+4)*5
    ('2 3 4 * +',   14.0),   # postfix: 2+(3*4)
    ('5 1 2 + 4 * + 3 -', 14.0),  # 5+((1+2)*4)-3
]
for expr, esperado in expresiones:
    resultado = evaluar_postfix(expr)
    assert resultado == esperado, f'Falló {expr!r}'
    print(f'{expr!r:25} = {resultado}  {"✓" if resultado==esperado else "✗"}')

# Shunting-Yard
infixes = ['3 + 4 * 5', '( 3 + 4 ) * 5', '2 + 3 * 4 - 1']
esperados_postfix = ['3 4 5 * +', '3 4 + 5 *', '2 3 4 * + 1 -']
for infix, esp in zip(infixes, esperados_postfix):
    postfix = infix_a_postfix(infix)
    valor = evaluar_postfix(postfix)
    assert postfix == esp, f'Postfix incorrecto para {infix!r}: {postfix!r}'
    print(f'Infix: {infix!r:25} → Postfix: {postfix!r:20} = {valor}')

# Casos de error
for malo in ('( 3 + 4', '3 + 4 )', '2+3'):
    try: infix_a_postfix(malo); raise AssertionError(f'{malo!r} debía fallar')
    except ValueError as e: print(f'  Error esperado en {malo!r}: {e}')

'3 4 + 5 *'               = 35.0  ✓
'2 3 4 * +'               = 14.0  ✓
'5 1 2 + 4 * + 3 -'       = 14.0  ✓
Infix: '3 + 4 * 5'               → Postfix: '3 4 5 * +'          = 23.0
Infix: '( 3 + 4 ) * 5'           → Postfix: '3 4 + 5 *'          = 35.0
Infix: '2 + 3 * 4 - 1'           → Postfix: '2 3 4 * + 1 -'      = 13.0
  Error esperado en '( 3 + 4': Paréntesis no balanceados: sobra '('
  Error esperado en '3 + 4 )': Paréntesis no balanceados: sobra ')'
  Error esperado en '2+3': Token no reconocido: '2+3' (separe los tokens con espacios)


## Actividad 4 — BFS (cola) vs DFS (pila) sobre la red de ciudades de Puno
Objetivo: comparar el orden de visita y visualizar los niveles de BFS.

In [4]:
from collections import deque

# Red vial simplificada de Puno — grafo no dirigido
CIUDADES = {
    0:'Puno', 1:'Juliaca', 2:'Ilave', 3:'Desaguadero',
    4:'Yunguyo', 5:'Juli', 6:'Lampa', 7:'Azangaro',
    8:'Huancane', 9:'Ayaviri'
}
GRAFO = {
    0:[1,2,5], 1:[0,6,7], 2:[0,3,4], 3:[2,4], 4:[2,3,5],
    5:[0,4], 6:[1], 7:[1,8,9], 8:[7], 9:[7]
}

def bfs(grafo, inicio, ciudades=None):
    """BFS con cola — retorna orden de visita y nivel de cada nodo."""
    visitados = {inicio}
    cola = deque([(inicio, 0)])  # (nodo, nivel)
    orden, niveles = [], {}
    while cola:
        nodo, nivel = cola.popleft()
        orden.append(nodo)
        niveles[nodo] = nivel
        for vecino in grafo[nodo]:
            if vecino not in visitados:
                visitados.add(vecino)
                cola.append((vecino, nivel+1))
    return orden, niveles

def dfs(grafo, inicio):
    """DFS con pila — retorna orden de visita."""
    visitados = set()
    pila = [inicio]
    orden = []
    while pila:
        nodo = pila.pop()
        if nodo not in visitados:
            visitados.add(nodo)
            orden.append(nodo)
            for vecino in reversed(grafo[nodo]):  # reversed para orden consistente
                if vecino not in visitados:
                    pila.append(vecino)
    return orden

orden_bfs, niveles = bfs(GRAFO, 0, CIUDADES)
orden_dfs = dfs(GRAFO, 0)

# Pruebas
assert orden_bfs == [0, 1, 2, 5, 6, 7, 3, 4, 8, 9]
assert orden_dfs == [0, 1, 6, 7, 8, 9, 2, 3, 4, 5]
assert len(orden_bfs) == len(orden_dfs) == len(GRAFO)   # el grafo es conexo

print('BFS desde Puno (nivel por nivel):')
for nivel in range(max(niveles.values())+1):
    nodos_nivel = [CIUDADES[n] for n,l in niveles.items() if l==nivel]
    print(f'  Nivel {nivel}: {nodos_nivel}')

print('\nBFS orden:', [CIUDADES[n] for n in orden_bfs])
print('DFS orden:', [CIUDADES[n] for n in orden_dfs])

BFS desde Puno (nivel por nivel):
  Nivel 0: ['Puno']
  Nivel 1: ['Juliaca', 'Ilave', 'Juli']
  Nivel 2: ['Lampa', 'Azangaro', 'Desaguadero', 'Yunguyo']
  Nivel 3: ['Huancane', 'Ayaviri']

BFS orden: ['Puno', 'Juliaca', 'Ilave', 'Juli', 'Lampa', 'Azangaro', 'Desaguadero', 'Yunguyo', 'Huancane', 'Ayaviri']
DFS orden: ['Puno', 'Juliaca', 'Lampa', 'Azangaro', 'Huancane', 'Ayaviri', 'Ilave', 'Desaguadero', 'Yunguyo', 'Juli']


## Actividad 5 — C++ (STL)
El código de esta actividad (balanceo con `std::stack`, BFS con `std::queue` y benchmark con `std::deque`)
está en el archivo **`Practica_S06_PilasColasDeque.cpp`** incluido en el .zip.

Compilar y ejecutar:
```bash
g++ -std=c++17 -O2 -Wall -o s06 Practica_S06_PilasColasDeque.cpp && ./s06
```

## Actividad 6 — Calculadora con historial
Objetivo: combinar balanceo de paréntesis, infix → postfix, evaluación postfix y una tabla hash de historial.

> Requiere haber ejecutado las celdas de las Actividades 2 y 3.

In [5]:
class CalculadoraConHistorial:
    """
    Mini-calculadora que:
    1) Valida la expresión (paréntesis balanceados)
    2) Convierte infix → postfix (Shunting-Yard)
    3) Evalúa el postfix
    4) Almacena en tabla hash {expresión: resultado}
    """
    def __init__(self):
        self.historial: dict = {}

    def calcular(self, expr: str) -> float:
        # 1. Validar paréntesis
        if not esta_balanceado(expr):
            raise ValueError(f'Paréntesis no balanceados: {expr!r}')

        # 2. Consultar historial (tabla hash)
        if expr in self.historial:
            print(f'  [cache hit] {expr!r} → {self.historial[expr]}')
            return self.historial[expr]

        # 3. Convertir y evaluar
        postfix = infix_a_postfix(expr)
        resultado = evaluar_postfix(postfix)

        # 4. Guardar en historial
        self.historial[expr] = resultado
        print(f'  [nuevo] {expr!r} → postfix={postfix!r} → {resultado}')
        return resultado

    def ver_historial(self):
        print('Historial de cálculos:')
        for expr, res in self.historial.items():
            print(f'  {expr!r:30} = {res}')

calc = CalculadoraConHistorial()
expresiones = [
    '3 + 4 * 5',
    '( 3 + 4 ) * 5',
    '3 + 4 * 5',   # repetida — debe usar cache
    '10 / 2 + 3',
]
resultados_esperados = [23.0, 35.0, 23.0, 8.0]
for e, esp in zip(expresiones, resultados_esperados):
    r = calc.calcular(e)
    assert r == esp, f'Falló {e!r}: {r} != {esp}'
    print(f'  Resultado: {r}')
assert len(calc.historial) == 3          # la expresión repetida no se guardó dos veces
calc.ver_historial()

  [nuevo] '3 + 4 * 5' → postfix='3 4 5 * +' → 23.0
  Resultado: 23.0
  [nuevo] '( 3 + 4 ) * 5' → postfix='3 4 + 5 *' → 35.0
  Resultado: 35.0
  [cache hit] '3 + 4 * 5' → 23.0
  Resultado: 23.0
  [nuevo] '10 / 2 + 3' → postfix='10 2 / 3 +' → 8.0
  Resultado: 8.0
Historial de cálculos:
  '3 + 4 * 5'                    = 23.0
  '( 3 + 4 ) * 5'                = 35.0
  '10 / 2 + 3'                   = 8.0
